In [8]:
from sentence_transformers import SentenceTransformer
import pandas as pd 
import numpy as np
import re 

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Loading model completed")
folder = "Data\\"
categories = pd.read_csv(folder + "amazon_final_category_mapping.csv")
categories["normalized_name"] = categories["category_name"].apply(normalize)
category_names = categories["normalized_name"].tolist()


categories["normalized_name"] = categories["category_name"].apply(normalize)
category_names = categories["normalized_name"].tolist()

category_embeddings = model.encode(
    categories["category_name"].tolist(),
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading model completed


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
def get_top_categories(llm_category, top_k=10):

    query_embedding = model.encode(
        [llm_category],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0].astype(np.float32)

    embedding_scores = category_embeddings @ query_embedding

    top_indices = np.argsort(-embedding_scores)[:top_k]

    results = []
    category_ids=[]
    print(f"Top {top_k} categories:")
    print("Category_id          Category_name")
    for idx in top_indices:
        print(int(categories.iloc[idx]["category_id"]),categories.iloc[idx]["category_name"])
        results.append({
            "category_id": int(categories.iloc[idx]["category_id"]),
            "category_name": categories.iloc[idx]["category_name"],
            "embedding_score": round(float(embedding_scores[idx]), 4),
        }) 
        category_ids.append( int(categories.iloc[idx]["category_id"]))
    
    return results,category_ids

In [3]:
from src.Search.Data_ingest import data_ingest 

products,categories = data_ingest()

Ingesting Data
Procesisng Data
Columns:
['Unnamed: 0', 'asin', 'title', 'category_name', 'category_id', 'stars', 'reviews', 'price', 'price_inr', 'imgUrl', 'productURL']

Number of products: 1393565


In [14]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
client_2=Groq(api_key=os.getenv("GROQ_API_KEY_2"))

# Models list for further changing 
# models = client.models.list()
# for model in models.data:
#     print(model.id)


In [15]:
from src.LLM.TEMPLATES.main_template import PROMPT_TEMPLATE 
from src.LLM.TEMPLATES.min_attribute import PROMPT_TEMPLATE_ATT_MIN 

In [16]:
PROMPT_TEMPLATE_PRODUCT = """
You are a product elaboration system designed to improve
semantic search and product category classification.

TASK:
Elaborate the given product name into a short, meaningful
description that helps a sentence embedding model understand
the product and identify relevant product categories.

INPUT:
{input_json}

RULES:
1. Identify the product type and its primary purpose or use.
2. Use simple, natural language and relevant contextual terms.
3. Do not invent features, materials, or specifications.
4. Do not list other products or unrelated categories.
5. The output must contain exactly one sentence.
6. Return valid JSON only, using the format shown below.

EXAMPLES:
Input: {"item": "Saree"}
Output: {"output": "A saree is a traditional women's ethnic garment worn by wrapping fabric around the body."}

Input: {"item": "Laptop"}
Output: {"output": "A laptop is a portable personal computer used for work, education, computing, and entertainment."}

Input: {"item": "Car Tyres"}
Output: {"output": "Car tyres are automotive components fitted to vehicle wheels to provide road grip, traction, and safe driving."}

OUTPUT FORMAT:
{
    "output": "One-sentence product description"
}

strictly the output should be in JSON FORMAT
"""

In [17]:
import json 

def attributes(template,promt_main):
    att_template={}
    att_template["user_promt"]=promt_main 
    att_template["items"]=[]
    for i,values in enumerate(template['items']):
        att_template["items"].append({})
        att_template["items"][i]['name']=values['name']
        att_template["items"][i]['preferences']=values['preferences']

    prompt =  PROMPT_TEMPLATE_ATT_MIN.replace(
        "{input_json}",
        json.dumps(att_template, ensure_ascii=False)
    )
    response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {
                    "role": "system",
                    "content": "Extract attributes accurately. Return JSON only."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
    )
    att_template=json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    for i,values in enumerate(template['items']):
        template["items"][i]['attributes']=att_template["items"][i]['attributes']
    return template

def elaborate_product(product_name):
    input_data = {"item": product_name}

    prompt = PROMPT_TEMPLATE_PRODUCT.replace(
        "{input_json}",
        json.dumps(input_data, ensure_ascii=False)
    )

    response = client_2.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Elaborate product names for semantic search. "
                    "Return JSON only."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    print("Elaborate Promt:",result["output"])
    return result["output"]
     

    
def adding_category(template):
    llm_items=template["items"]
    for i in range(len(llm_items)):
            llm_category=template["items"][i]["name"]
            elaborate_promt=elaborate_product(llm_category)
            results,category_ids=get_top_categories(elaborate_promt)
            template["items"][i]["category_id"]=category_ids 
    return template

def extract_requirements(user_prompt):
    prompt = PROMPT_TEMPLATE.replace("{user_prompt}", user_prompt)

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": "Extract requirements accurately. Return JSON only."
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    template=json.loads(response.choices[0].message.content)# pyright: ignore[reportArgumentType]
    template=attributes(template,user_prompt)
    main_template=adding_category(template)

    
    return main_template

In [18]:
prompt="Show me some sarees(Saree is a Women Clothing)"
main_template=extract_requirements(prompt)
print(main_template)

Elaborate Promt: Sarees are traditional women's garments made of a long piece of fabric that is wrapped around the body and draped over the shoulder.
Top 10 categories:
Category_id          Category_name
116 Women's Clothing
118 Women's Handbags
120 Women's Accessories
110 Men's Clothing
91 Girls' Clothing
102 Garment Bags
122 Women's Shoes
2 Fabric Decorating
121 Women's Watches
84 Boys' Clothing
{'items': [{'name': 'Sarees', 'keywords': ['saree'], 'preferences': [], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'target_user': ['women']}, 'category_id': [116, 118, 120, 110, 91, 102, 122, 2, 121, 84]}]}


In [19]:
main_template

{'items': [{'name': 'Sarees',
   'keywords': ['saree'],
   'preferences': [],
   'budget': None,
   'min_rating': None,
   'min_reviews': None,
   'attributes': {'target_user': ['women']},
   'category_id': [116, 118, 120, 110, 91, 102, 122, 2, 121, 84]}]}

In [20]:

import re
import pandas as pd
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"[-_/]+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def word_variants(word):
    variants = {word}

    # Common English plural forms
    if len(word) > 3 and word.endswith("ies"):
        variants.add(word[:-3] + "y")

    if len(word) > 3 and word.endswith("es"):
        variants.add(word[:-2])

    if len(word) > 2 and word.endswith("s") and not word.endswith("ss"):
        variants.add(word[:-1])

    return variants

def matches_term(title, term):
    title_words = normalize_text(title).split()
    term_words = normalize_text(term).split()

    if not term_words:
        return False

    # Match a word or phrase, allowing common plural differences
    title_variants = [word_variants(w) for w in title_words]
    term_variants = [word_variants(w) for w in term_words]

    for i in range(len(title_words) - len(term_words) + 1):
        matched = all(
            title_variants[i + j] & term_variants[j]
            for j in range(len(term_words))
        )
        if matched:
            return True

    return False

def search_products(domain, item):
    domain = domain.copy()
    titles = domain["title"].fillna("").map(normalize_text)

    keywords = item.get("keywords") or []
    preferences = item.get("preferences") or []

    # Require ALL keywords to match (AND logic)
    valid_keywords = [
        normalize_text(k) for k in keywords if normalize_text(k)
    ]

    if valid_keywords:
        mask = pd.Series(True, index=domain.index)

        for keyword in valid_keywords:
            keyword_mask = pd.Series(
                [
                    matches_term(title, keyword)
                    for title in titles
                ],
                index=domain.index
            )
            mask &= keyword_mask

        domain = domain.loc[mask].copy()
        titles = titles.loc[domain.index]

    # Rank products by the number of matched preferences
    valid_preferences = [
        normalize_text(p)
        for p in preferences if normalize_text(p)
    ]

    if valid_preferences:
        domain["preference_matches"] = [
            sum(
                matches_term(title, pref)
                for pref in valid_preferences
            )
            for title in titles
        ]

        domain = domain.sort_values(
            "preference_matches",
            ascending=False,
            kind="stable"
        )

    return domain


In [24]:

import pandas as pd

class Searching:
    def __init__(self,products,categories):
        print("Getting all the Data")
        self.products, self.categories = products,categories
        self.nodes_explored = 0
        self.backtracks = 0

    def get_domains(self, requirements):
        domains = {}
        total_possible_combinations=1
        for item in requirements["items"]:
            item_name = item["name"]
            category_ids = item["category_id"]

            if not isinstance(category_ids, list):
                category_ids = [category_ids]

            products = self.products
            domain = products[
                products["asin"].notna()&
                products["title"].notna()&
                products["category_id"].isin(category_ids) &
                products["category_name"].notna() &
                products["price_inr"].notna() &
                products["stars"].notna() &
                products["reviews"].notna()
            ]


            if item.get("budget") is not None:
                domain = domain[domain["price_inr"] <= item["budget"]]

            if item.get("min_rating") is not None:
                domain = domain[
                    domain["stars"] >= item["min_rating"]
                ]

            if item.get("min_reviews") is not None:
                domain = domain[
                    domain["reviews"] >= item["min_reviews"]
                ]

            #Searching According to the Name 
            # if item.get("keywords") is not None:
            #     for keyword in item["keywords"]:
            #         domain = domain[domain["title"].astype(str).str.contains(keyword, case=False, na=False)] 

            # if item.get("preferences") is not None:
            #     for preference in item["preferences"]:
            #         domain = domain[domain["title"].astype(str).str.contains(preference, case=False, na=False)]

            domain = search_products(domain, item)
            domains[item_name] = domain 
            total_possible_combinations*=len(domain)
            print(item_name, "domain size:", len(domain))

        return domains,total_possible_combinations

    def is_consistent(self, assignment, new_product, item, requirements):
        for product in assignment.values():
            if product["asin"] == new_product["asin"]:
                return False

        if item.get("budget") is not None:
            if new_product["price_inr"] > item["budget"]:
                return False

        if item.get("min_rating") is not None:
            if new_product["stars"] < item["min_rating"]:
                return False

        if item.get("min_reviews") is not None:
            if new_product["reviews"] < item["min_reviews"]:
                return False

        total_cost = new_product["price_inr"]
        for product in assignment.values():
            total_cost += product["price_inr"]

        if requirements.get("budget") is not None:
            if total_cost > requirements["budget"]:
                return False

        return True

   
    def backtracking_search(
        self, variables, domains, assignment, requirements, items,
        solutions, max_solutions
    ):
        if len(solutions) >= max_solutions:
            return

        if len(assignment) == len(variables):
            solutions.append(assignment.copy())
            return

        variable = variables[len(assignment)]
        item = items[variable]

        for index in domains[variable].index:
            if len(solutions) >= max_solutions:
                return

            self.nodes_explored += 1
            product = self.products.loc[index]

            if self.is_consistent(
                assignment, product, item, requirements
            ):
                assignment[variable] = product

                self.backtracking_search(
                    variables, domains, assignment, requirements, items,
                    solutions, max_solutions
                )

                del assignment[variable]


    def finding_products(self, requirements):
        print(requirements)

        variables = [item["name"] for item in requirements["items"]]
        items = {item["name"]: item for item in requirements["items"]}

        self.nodes_explored = 0
        self.backtracks = 0

        domains,total_possible_combinations = self.get_domains(requirements)
        print("Total Possible Combiantions:",total_possible_combinations)
        top_k_solutions=10

        # Fail early if any item has an empty domain
        if any(domains[v].empty for v in variables):
            print("NO FEASIBLE SOLUTION: One or more domains are empty.")
            return None

        solutions = []

        self.backtracking_search(
            variables, domains, {}, requirements, items,
            solutions, max_solutions=top_k_solutions
        )

        if not solutions:
            print("NO FEASIBLE SOLUTION FOUND")
        else:
            for i, solution in enumerate(solutions, 1):
                print(f"\nSolution {i}")

                total_cost = 0

                for variable, product in solution.items():
                    print("Item:", variable)
                    print("Product:", product["title"])
                    print("Price:", product["price_inr"])
                    print("Category ID:", product["category_id"])
                    print("Category Name:", product["category_name"])
                    total_cost += product["price_inr"]

                print("------------------------")
                print("Total cost:", total_cost)


        print("Budget:", requirements.get("budget"))
        print("Nodes explored:", self.nodes_explored)
        print("Backtracks:", self.backtracks)

        return solutions


In [25]:


obj=Searching(products,categories)

Getting all the Data


In [29]:
# prompt="show me black hiking shoes under 50 with white t-shirt with  L size with black shorts with L size"
prompt=str(input("Enter the prompt"))
print(prompt)
main_template=extract_requirements(prompt)
print(main_template)
solutions=obj.finding_products(main_template)

show me red saree with black heels
Elaborate Promt: Sarees are traditional women's ethnic garments that consist of a long piece of fabric draped around the body and are worn for cultural, ceremonial, and everyday occasions.
Top 5 categories:
Category_id          Category_name
116 Women's Clothing
122 Women's Shoes
110 Men's Clothing
91 Girls' Clothing
120 Women's Accessories
Elaborate Promt: Heels are women’s shoes with elevated heels designed for fashion, style, and formal occasions.
Top 5 categories:
Category_id          Category_name
122 Women's Shoes
97 Girls' Shoes
114 Men's Shoes
116 Women's Clothing
90 Boys' Shoes
{'items': [{'name': 'Sarees', 'keywords': ['saree'], 'preferences': ['red'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['red']}, 'category_id': [116, 122, 110, 91, 120]}, {'name': 'Heels', 'keywords': ['shoes'], 'preferences': ['black'], 'budget': None, 'min_rating': None, 'min_reviews': None, 'attributes': {'color': ['black']}, '